
# Important Update

This notebook was updated for compatibility with newer LangChain versions.

Key fixes:
- Updated deprecated imports
- Fixed OpenAIEmbeddings usage
- Fixed FAISS compatibility issues
- Updated RetrievalQA invocation style

Before running the notebook again:
1. Delete old `faiss_index` folder
2. Recreate embeddings and vector store from scratch


In [ ]:
# “Let me ask you something — How many of you have used ChatGPT or Gemini or Copilot and realized that sometimes it gives beautifully structured answers, but... they’re confidently wrong?”
# >“Yes!” / “All the time.” / “It sounds perfect but is factually incorrect.”

# Large Language Models, or LLMs, are like very knowledgeable storytellers. They’re trained on massive text data from the internet, but once the training is complete, they stop learning. They don’t know your private data, your company’s documents, or anything that happened after their cutoff date.
# So, when you ask a question beyond their memory, they do what humans often do under pressure — they guess elegantly.
# This guessing problem is what we call ‘hallucination’.
# And that’s not acceptable in domains like healthcare, banking, or legal — where accuracy matters more than eloquence.

In [ ]:
# What Are Hallucinations in LLMs?

# Hallucinations in LLMs refer to factually incorrect, fabricated, or nonsensical outputs that sound plausible or fluent but are not grounded in reality or reliable data.

# In simpler terms:
# An LLM “hallucinates” when it confidently makes something up.

# These hallucinations can be factual, logical, or contextual errors.

In [ ]:
# Types of Hallucinations

# 1. Factual Hallucinations: When the model provides information that is incorrect or fabricated. For example, stating that "The Eiffel Tower is located in Berlin" is a factual hallucination.

# 2. Fabricated Sources: When the model cites non-existent articles, studies, or references to support its claims. The model invents fake citations, links, or studies.
# eg: According to the 2023 Harvard Study on Coffee by Dr. Jane Smith…” (which doesn’t exist)

# 3. Logical Inconsistencies: When the model produces outputs that contradict itself or contain logical fallacies. The reasoning chain or math is inconsistent.
# eg: “The sun rises in the west and sets in the east.”
# eg: If a car travels 60 km in 2 hours, its speed is 15 km/h.

# 4. Contextual Hallucinations: When the model misinterprets the context of a question or conversation, leading to irrelevant or incorrect responses. The model misinterprets user intent or invents missing context.
# eg: If asked about "Apple" in a tech context, the model talks about the fruit instead of the company.
# eg: When asked “Summarize this article,” but no article was given — it fabricates one.

# 5. Temporal Hallucinations: When the model provides information that is outdated or incorrect based on the current time frame. The model invents events or facts that are not aligned with the current date.
# eg: Stating that "Barack Obama is the current president of the United States".
# eg: Claiming that "The 2024 Summer Olympics were held in Tokyo."

# 6. Multimodal Hallucinations: In models that process multiple types of data (like text and images), hallucinations can occur when the model misinterprets or fabricates information across these modalities. The model invents details about images or videos it analyzes.
# eg: Describing objects in an image that are not actually present.
# eg: Claiming a video shows an event that never occurred.

In [ ]:
# Why Do Hallucinations Happen?

# 1. Training Objective (Next Token Prediction)
# LLMs are trained to predict the next word that sounds right, not necessarily what’s true.
# → They optimize for linguistic plausibility, not factual accuracy.

# 2. Data Limitations
# Training data can contain errors or outdated facts.
# Some topics (e.g., niche scientific findings) may be underrepresented.

# 3. Overgeneralization & Pattern Completion
# LLMs infer patterns from partial data and “fill in the blanks,” which can lead to confident but wrong completions.

# 4. Prompt Ambiguity
# If the user’s query is vague, the model might make assumptions or fabricate context.

# 5. Lack of External Grounding
# The model doesn’t inherently know the current state of the world — unless connected to verified sources or retrieval systems.

In [ ]:
# Examples of LLM Hallucination

# Example 1: Fake Citation
# Prompt: “Give me a recent study on quantum batteries.”
# Hallucinated Output: “A 2024 study by the University of Cambridge found that quantum batteries can charge 50% faster.”
# Reality: No such study exists.

# Example 2: Invented Historical Fact
# Prompt: “Who was the president of the USA in 1850?”
# Output: “Andrew Jackson.”
# Reality: It was Millard Fillmore.

# Example 3: Miscalculated Logic
# Prompt: “If each person eats 2 slices and there are 8 people, how many pizzas with 8 slices each are needed?”
# Output: “You’ll need 1 pizza.”
# Reality: 8×2=16 slices → 2 pizzas needed.

In [ ]:
# How to Reduce or Prevent Hallucinations

# 1. Retrieval-Augmented Generation (RAG)
# Combine the LLM with live access to trusted data sources (databases, documents, the web).
# The model retrieves relevant information before generating an answer.
# Example: ChatGPT with “Browse with Bing” or an enterprise LLM connected to a company knowledge base.
# Result: Grounded responses, less guessing.

# 2. Fact-Checking and Verification Layers
# Post-process the model’s output using:
# - External APIs (Wikipedia, PubMed, etc.)
# - Rule-based or smaller models specialized in fact-checking.

# 3. Prompt Engineering
# Craft prompts that:
# - Ask for sources or uncertainty estimates.
# - Encourage “honest reasoning.”
# Example:
# Instead of → “Give me the answer,”
# Try → “If you’re unsure, say so. Cite reliable sources where possible.”

# 4. Training Improvements
# a. RLHF (Reinforcement Learning from Human Feedback): Trains models to prefer truthful, high-quality responses.
# b. RLAIF (AI Feedback): Uses smaller models or automated systems to penalize hallucinations.
# c. Synthetic Negative Examples: Fine-tune with examples of hallucinated vs. grounded outputs.

# 5. Transparency / Uncertainty Estimation
# Have the model express confidence levels or cite reasoning.
# E.g., “I’m 70% confident that the capital of Australia is Canberra.”

# 6. User Awareness and Verification
# Encourage users to:
# Double-check critical outputs (especially legal, medical, or financial info).
# Use LLMs as assistants, not authorities.

| Problem            | Why It Happens                    | How to Mitigate                          |
| ------------------ | --------------------------------- | ---------------------------------------- |
| Factual errors     | Predictive, not factual objective | Use RAG, connect to verified data        |
| Fabricated sources | Pattern completion                | Verify citations, use citation databases |
| Logical mistakes   | Limited reasoning                 | Chain-of-thought or verification models  |
| Overconfidence     | No built-in uncertainty           | Add confidence estimates or disclaimers  |


In [ ]:
# Imagine you’re an interviewer who’s about to take a technical interview. You haven’t prepared in months. Would you rather:
# A) Trust your memory from last year’s prep, or
# B) Quickly Google the latest concepts before answering?
# >Option B — I’ll definitely check the latest ones

# That’s exactly what RAG does.
# Instead of relying only on the model’s old ‘memory,’ RAG lets the model retrieve the most recent and relevant information first — and then generate an answer using both what it already knows and what it just fetched.

In [ ]:
# Imagine you’re in a large organization. You want to ask a chatbot:
# ‘What’s our leave policy after maternity leave?’ or
# ‘Summarize last quarter’s performance from our internal reports.’
# Do you think ChatGPT, as it is, can answer that?”
# >No — because it doesn’t have access to internal data.

# ChatGPT doesn’t have your internal policies or reports — but a RAG-based chatbot can.
# It connects to your company’s documents, retrieves the relevant chunks from them, and then uses the LLM to compose a human-like, contextually accurate answer.

# LLM → knows language patterns
# RAG → adds your data
# Together → you get accurate, context-aware, and grounded answers.

In [ ]:
# Suppose your company wants a chatbot that understands your own documents, policies, and reports.
# Would you rather:
# A) Train or fine-tune your own LLM with all that data,
# or
# B) Plug your data into ChatGPT or an open-source model that already exists and just let it ‘refer’ to it when needed?
# >Option B sounds easier.” / “Fine-tuning sounds expensive"

# Fine-tuning or retraining a large language model is extremely expensive.
# It requires massive computational resources, time, and expertise.
# Plus, every time your documents update, you’d need to retrain or fine-tune again.
# RAG sidesteps this by keeping the LLM fixed and simply updating the retrieval database as needed.
# This makes it much more efficient and cost-effective for integrating private or dynamic data.

| Factor           | Fine-Tuning Approach             | RAG Approach               |
| :--------------- | :------------------------------- | :------------------------- |
| **Cost**         | Very high (compute + retraining) | Low (no model training)    |
| **Time**         | Weeks or months                  | Hours or days              |
| **Data Updates** | Needs retraining                 | Instantly reflected        |
| **Flexibility**  | Fixed to trained data            | Dynamic and updatable      |
| **Risk**         | Possible overfitting             | None (data stays external) |


In [ ]:
# So if your HR policy or customer data changes next week,
# in fine-tuning, you’d have to train again, costing you time and GPU cycles.
# But with RAG, you just update your vector database.
# The model stays the same — it simply fetches new facts before answering.

# Fine-tuning is like engraving information on stone — costly and hard to change.
# RAG is like reading from a dynamic library — cheaper, flexible, and always current.

In [ ]:
# Let’s take a real example.
# Say you have 10,000 internal documents and you want a chatbot that can answer from them.

| Step           | Fine-Tuning Cost (approx.) | RAG Cost (approx.)              |
| -------------- | -------------------------- | ------------------------------- |
| Model Training | $3,000–$10,000             | $0                              |
| GPU Hosting    | $1,000/month               | $100–$200/month (for vector DB) |
| Data Update    | Retraining again ($3K+)    | Just re-embed ($50–$100)        |
| Total (Year 1) | ~$20,000+                  | ~$1,000–$2,000                  |


In [ ]:
# https://miro.medium.com/v2/resize:fit:1400/0*9G8VvkAZUNv-vQYr.png

![image.png](attachment:image.png)

---

## **Problem Statement**

In real-world applications, organizations often store important information in documents such as **policy manuals, handbooks, or technical PDFs**. Traditional language models like GPT-4 do not have access to these documents unless explicitly given the data.

This project demonstrates how to build a **Retrieval-Augmented Generation (RAG)** system using LangChain and OpenAI. It allows the language model to **search inside a PDF document**, retrieve relevant information, and generate accurate, context-specific answers.

---

## **Objectives**

1. Load content from a PDF file into a LangChain-compatible format.
2. Split the PDF content into manageable chunks for efficient vector storage.
3. Generate embeddings for each chunk using OpenAI Embeddings.
4. Store and search the chunks using the FAISS vector store.
5. Use GPT-4 Turbo as the language model to generate context-aware answers using retrieved chunks.
6. Ensure that the implementation uses the latest LangChain and OpenAI APIs without any deprecation warnings.

---

## **Expected Outcomes**

- A working RAG pipeline that uses PDF content as a knowledge base.
- The system can answer user queries based on the information stored inside the PDF.
- It will retrieve relevant passages from the PDF and use them to generate answers via GPT-4.
- The setup will be modular, extensible, and aligned with the latest LangChain version.

---

In [2]:
import os # operating system library to interact with the environment variables and file paths
from dotenv import load_dotenv # to load environment variables from a .env file
from langchain.document_loaders import PyPDFLoader # a class used to load text from PDF files

# Imports a powerful text-splitting tool that breaks large documents into manageable chunks for embedding.
# LLMs have context length limits, so splitting documents into chunks is essential.
# Why recursive?
# It tries to split at logical boundaries (like paragraphs, sentences, etc.) before falling back to raw character limits.
from langchain.text_splitter import RecursiveCharacterTextSplitter

# OpenAIEmbeddings - to convert text into vector format using OpenAI’s embedding API
#from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chat_models import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings, OpenAIEmbeddings

from langchain_community.vectorstores import FAISS # a fast in-memory vector search engine used for semantic search.

"""Imports RetrievalQA, a chain that connects:
-A retriever (like FAISS)
-A language model (like GPT): To answer questions using retrieved context.

This is the heart of a RAG system — combining search + generation"""
from langchain.chains import RetrievalQA

In [3]:
chat_model = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo")


In [ ]:
# Step 1: Load environment variables (.env should have your OPENAI_API_KEY)
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
# %pip install pypdf -q

In [10]:
# Step 2: Load PDF document (Replace with your actual file path)
pdf_path = "docs/company_policy.pdf"
# Make sure this file exists

try:
    loader = PyPDFLoader(pdf_path) # Load the PDF using PyPDFLoader
    documents = loader.load() # Extracts the text content from the PDF and stores it in 'documents'
except Exception as e:
    print(f"Error loading PDF: {e}")
else:
    print(f"Loaded {len(documents)} pages from the PDF.")

Loaded 24 pages from the PDF.


In [11]:
# import os

# print("Current Working Directory:\n")
# print(os.getcwd())

# print("\nChecking file existence:\n")
# print(os.path.exists("work/output_model_mem/docs/company_policy.pdf"))


In [ ]:
# documents

In [ ]:
"""
1. PDF Loaders
Loader =>	Use When...	=> Handles Images/Tables?	=> Code Example
PyPDFLoader	Simple text-based PDFs    => No	=> loader = PyPDFLoader("path/to/pdf")
PDFMinerLoader	Needs layout-aware text extraction => No	=> loader = PDFMinerLoader("file.pdf")
PDFPlumberLoader	=> Needs table extraction => Basic tables	=> loader = PDFPlumberLoader("file.pdf")
UnstructuredPDFLoader	=> Complex structure, mixed text, tables, images => Yes (OCR + layout)    => loader = UnstructuredPDFLoader("file.pdf")
PyMuPDFLoader	Text + metadata (fast, accurate) => (Limited)	=> loader = PyMuPDFLoader("file.pdf")

2. Word Documents (DOC / DOCX)
Loader	=> Use When...	=> Code
UnstructuredWordDocumentLoader	You have .docx files	=> loader = UnstructuredWordDocumentLoader("file.docx")
Docx2txtLoader	Basic text extraction from .docx	=> loader = Docx2txtLoader("file.docx")

3. Excel Files (XLS / XLSX)
Loader	Use When...	Code
UnstructuredExcelLoader	Full sheet extraction	=> loader = UnstructuredExcelLoader("file.xlsx")
PandasExcelLoader	Structured loading using pandas	=> loader = PandasExcelLoader("file.xlsx")

4. CSV Files
Loader	Use When...	Code
CSVLoader	Simple CSV reading	=> loader = CSVLoader("file.csv")
PandasCSVLoader	Use Pandas for control & filtering	=> loader = PandasCSVLoader("file.csv")

5. JSON / JSONL
Loader	Use When...	Code
JSONLoader	Simple .json file with nested fields    => loader = JSONLoader(file_path="file.json", jq_schema='.data[].text', text_content=False)

6. Web & HTML
Loader	Use When...	Code
WebBaseLoader	Load content from a URL	WebBaseLoader("https://example.com")
UnstructuredHTMLLoader	Clean up raw HTML structure	UnstructuredHTMLLoader("file.html")
"""
"""
Load PDF with Tables using PDFPlumberLoader
from langchain.document_loaders import PDFPlumberLoader
loader = PDFPlumberLoader("report_with_tables.pdf")
documents = loader.load()

Load Excel Sheet using UnstructuredExcelLoader
from langchain.document_loaders import UnstructuredExcelLoader
loader = UnstructuredExcelLoader("financials.xlsx")
documents = loader.load()

Load JSON File
from langchain.document_loaders import JSONLoader
loader = JSONLoader(file_path="data.json", jq_schema=".records[].summary", text_content=True)
documents = loader.load()

Load DOCX File
from langchain.document_loaders import UnstructuredWordDocumentLoader
loader = UnstructuredWordDocumentLoader("policy.docx")
documents = loader.load()
"""

"""
How to Choose the Right Loader?
| Content Type               | Recommended Loader                                          | Why                           |
| -------------------------- | ----------------------------------------------------------- | ----------------------------- |
| Simple PDFs                | `PyPDFLoader`                                               | Fast, reliable                |
| PDFs with tables           | `PDFPlumberLoader`                                          | Can extract table content     |
| PDFs with images / scanned | `UnstructuredPDFLoader`                                     | Uses OCR and layout modeling  |
| Word / Excel               | `UnstructuredWordDocumentLoader`, `UnstructuredExcelLoader` | Best structure preservation   |
| CSV / JSON                 | `PandasCSVLoader`, `JSONLoader`                             | Allows flexible preprocessing |
"""

'\nHow to Choose the Right Loader?\n| Content Type               | Recommended Loader                                          | Why                           |\n| -------------------------- | ----------------------------------------------------------- | ----------------------------- |\n| Simple PDFs                | `PyPDFLoader`                                               | Fast, reliable                |\n| PDFs with tables           | `PDFPlumberLoader`                                          | Can extract table content     |\n| PDFs with images / scanned | `UnstructuredPDFLoader`                                     | Uses OCR and layout modeling  |\n| Word / Excel               | `UnstructuredWordDocumentLoader`, `UnstructuredExcelLoader` | Best structure preservation   |\n| CSV / JSON                 | `PandasCSVLoader`, `JSONLoader`                             | Allows flexible preprocessing |\n'

In [14]:
# Step 3: Split PDF text into smaller chunks
# https://miro.medium.com/v2/resize:fit:1400/1*yfeUrFCr9oEVZofS8TvDEg.png
# RecursiveCharacterTextSplitter tries to split at logical boundaries (paragraph → sentence → word → character) — hence, recursive
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, # chunk_size=500: Each chunk will be up to 500 characters long.
    chunk_overlap=100 # chunk_overlap=100: The last 100 characters of one chunk are repeated in the next chunk for context continuity.
)
chunks = text_splitter.split_documents(documents) # take 24 pages and split into smaller chunks

# Print number of chunks created
print(f"Splitted into {len(chunks)} chunks.")
print()

# Print first chunk for inspection
print(chunks[0])

Splitted into 122 chunks.

page_content='SPIL Corporate HR Policies  \n \n \nSIRCA PAINTS INDIA LTD \nNEW DELHI  \n \n \n \n \nCORPORATE  \n  HUMAN RESOURCES \nPOLICIES & MANUALS' metadata={'source': 'docs/company_policy.pdf', 'page': 0}


In [15]:
# Print last chunk for inspection
print(chunks[-1])


page_content='Section : 20  Review and Amendment  \nManagement shall review this policy periodically and amendments required, if any shall be made \naccordingly.  \nSection : 21 Residual Power \nThis policy is basically guidelines and the management reserves the right to withdraw / modify to \nsuit organization’s philosophy at any time without assigning any reason whatsoever. \nEFFECTIVE \nCommencement Of Policy  August 21, 2018  \n \n \n \nApproved By : ___________SD/-_______________ \nMr Sanjay Agarwal - CMD' metadata={'source': 'docs/company_policy.pdf', 'page': 23}


In [ ]:
"""
Why Is Chunking So Important in RAG?
LLMs (like GPT-4) have a limited context window (e.g., 8K or 32K tokens), so you can't send the whole document.
Chunking allows:
-Semantic search over smaller pieces of the document
-Better matching with the user's query
-Faster retrieval, better accuracy

Tradional vs Semantic Search example
Traditional Search: Keyword-based search that looks for exact matches of words or phrases in documents.
Semantic Search: Understands the meaning and context of the query to find relevant documents, even if they don't contain the exact keywords.
Example:
Query: "What is the company's leave policy after maternity leave?"
Traditional Search Result: Might return documents that contain the exact phrase "maternity leave policy."
Semantic Search Result: Would return documents that discuss leave policies, maternity benefits, and related topics, even if they don't use the exact phrase.

Common Chunking Strategies (Used in Industry)
| Splitter Class                          | Description                                           | Use Case                                           |
| --------------------------------------- | ----------------------------------------------------- | -------------------------------------------------- |
| `RecursiveCharacterTextSplitter`        | Splits at paragraphs → sentences → words → characters | Best for generic documents (PDFs, policies, books) |
| `CharacterTextSplitter`                 | Splits at fixed character limits (naive)              | Simple logs, structured text                       |
| `TokenTextSplitter`                     | Splits by token count (uses tokenizer like tiktoken)  | Precise control for GPT-3.5/4                      |
| `SentenceTransformersTokenTextSplitter` | Aware of sentence boundaries + tokens                 | Best for multilingual and NLP-heavy documents      |
| `MarkdownHeaderTextSplitter`            | Splits based on Markdown headers                      | Technical docs, blog posts, notebooks              |
| `HTMLHeaderTextSplitter`                | Splits based on HTML tags                             | Websites, web-scraped data                         |
| `Language` Splitters                    | Code-aware (Python, JS, etc.)                         | Splits by function/class — great for code docs     |


How to Choose chunk_size and chunk_overlap?
chunk_size: How big each chunk is (in characters or tokens)
| Scenario                       | Recommended Chunk Size               |
| ------------------------------ | ------------------------------------ |
| Short emails, chat transcripts | 300–500 chars                        |
| Policies, contracts, PDFs      | 500–1000 chars                       |
| Technical manuals or FAQs      | 800–1500 chars                       |
| Code or JSON files             | 100–300 chars (or by function/class) |
| Scientific papers (dense info) | 1000–1500 chars or 256–512 tokens    |
| For OpenAI GPT-4 (8k model)    | Chunk size ≤ 1000 tokens             |
| For GPT-4 Turbo (128k model)   | Chunk size ≤ 3000 tokens             |

chunk_overlap: How much to repeat from previous chunk
| Purpose                              | Suggested Overlap                                      |
| ------------------------------------ | ------------------------------------------------------ |
| Maintain sentence continuity         | 10–20% of chunk\_size (e.g., 100 chars for 500 chunks) |
| Prevent cutoff of entity names       | 100–150 chars                                          |
| Use with semantic search (retriever) | 10–20%                                                 |
| Use with QA or summarization         | 100–200 chars                                          |

Example Best Practices:
Contract Documents:
RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
Contracts often have long clauses — more overlap helps preserve clause boundaries.

FAQs or Web Pages:
RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)

Code Files:
PythonCodeTextSplitter(chunk_lines=30, chunk_overlap=10)
"""

'\nWhy Is Chunking So Important in RAG?\nLLMs (like GPT-4) have a limited context window (e.g., 8K or 32K tokens), so you can\'t send the whole document.\nChunking allows:\n-Semantic search over smaller pieces of the document\n-Better matching with the user\'s query\n-Faster retrieval, better accuracy\n\nTradional vs Semantic Search example\nTraditional Search: Keyword-based search that looks for exact matches of words or phrases in documents.\nSemantic Search: Understands the meaning and context of the query to find relevant documents, even if they don\'t contain the exact keywords.\nExample:\nQuery: "What is the company\'s leave policy after maternity leave?"\nTraditional Search Result: Might return documents that contain the exact phrase "maternity leave policy."\nSemantic Search Result: Would return documents that discuss leave policies, maternity benefits, and related topics, even if they don\'t use the exact phrase.\n\nCommon Chunking Strategies (Used in Industry)\n| Splitter Cla

In [17]:
# Step 4: Create embeddings for each chunk using OpenAI
# https://platform.openai.com/docs/pricing
embeddings = OpenAIEmbeddings() # text-embedding-ada-002
# embeddings = OpenAIEmbeddings(model="text-embedding-ada-002", api_key=openai_api_key)

/voc/work/.local/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The class `langchain_community.embeddings.openai.OpenAIEmbeddings` was deprecated in langchain-community 0.1.0 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  warn_deprecated(


In [18]:
# Print the embedded model name
print(embeddings.model)

text-embedding-ada-002


In [ ]:
# print(embeddings)

In [27]:
# Print the embedding of the 0th chunk
#print(embeddings.embed_query(chunks[0].page_content))

# # Print the length of the 0th embedding
# print(len(embeddings.embed_query(chunks[0].page_content)))

# # Print the length of the 110th embedding
# print(len(embeddings.embed_query(chunks[110].page_content)))

In [ ]:
# %pip install faiss-cpu -q

In [20]:
from langchain_community.vectorstores import FAISS

In [29]:
# Step 5: Store chunks in FAISS vector store
"""Converts a list of text chunks into vectors using the embeddings model (e.g., OpenAIEmbeddings).
Stores those vectors inside a FAISS vector store
Returns a vectorstore object that supports fast semantic search (using cosine or inner product similarity)"""
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [ ]:
# Save the FAISS index to disk
# vectorstore.save_local("faiss_index")

In [ ]:
# Load the FAISS index from disk (with allow_dangerous_deserialization=True)
# Note: This is safe because we created this index ourselves

# vectorstore = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
# print("Vector store loaded from disk successfully.")
# print(f"Number of vectors in the vector store: {vectorstore.index.ntotal}")

In [ ]:

"""
What is a Vector Store?
A vector store (or vector database) is a specialized storage engine that:
-Stores high-dimensional embeddings (vectors)
-Supports similarity search (e.g., cosine similarity, dot product)
-Returns the most relevant stored chunks given a query

Used in:
-Semantic search
-RAG systems (retrieval-augmented generation)
-Recommendation engines
-Image/audio search

Common Vector Stores in the Industry (Open-source + Paid)
| Name                     | Type                     | Open Source | Cloud Hosted           | Common Use Case                       | Notes                                   |
| ------------------------ | ------------------------ | ----------- | ---------------------- | ------------------------------------- | --------------------------------------- |
| **FAISS**                | In-memory                | Yes         | No                     | Fast local prototyping                | Lightweight, fast, no persistence       |
| **Chroma**               | Embedded / local         | Yes         | Yes (through services) | Quick, simple RAG systems             | Comes with LangChain by default         |
| **Weaviate**             | Cloud-native             | Yes         | Yes                    | Enterprise-scale semantic search      | Supports hybrid search (text + vector)  |
| **Pinecone**             | Cloud-hosted             | No          | Yes                    | Scalable production RAG               | High availability, fast search          |
| **Qdrant**               | Cloud-native             | Yes         | Yes                    | High-perf vector search with metadata | Good for hybrid + multi-modal           |
| **Milvus**               | Cloud/on-prem            | Yes         | Yes                    | AI at scale (video, image, text)      | GPU acceleration available              |
| **ElasticSearch + k-NN** | General-purpose + plugin | Partially   | Yes                    | Enterprises with existing ES stack    | Not optimized for dense vectors         |
| **Redis + Redis Vector** | General-purpose + plugin | Yes         | Yes                    | Realtime vector search                | Limited advanced vector search features |

Vector Store Comparisons: Speed, Cost, and Use Cases
1. FAISS
Speed: Extremely fast (C++ backend, in-memory)
Cost: Free (open source)
Storage: In-memory only (unless manually persisted)
Best for: Prototyping, small datasets, no need for persistence
Limitations: Not suitable for production unless wrapped in a persistence layer

2. Pinecone
Speed: High (vector indexes stored in cloud)
Cost: Paid, with free tier (based on vector count + queries)
Best for: Scalable production RAG with real-time search
Organizations: Used by startups to large SaaS platforms
Strengths: Metadata filtering, hybrid search, horizontal scaling

3. Qdrant
Speed: Very good (Rust backend)
Cost: Free self-hosted; cloud pricing available
Best for: Production systems where filtering and custom payloads are needed
Organizations: ML teams, research labs, open-source apps
Strengths: JSON metadata, filtering, search inside images, multi-modal

4. Weaviate
Speed: Very good
Cost: Free (local) + paid cloud tier
Best for: NLP apps, semantic search, hybrid search
Strengths: GraphQL API, hybrid (BM25 + dense), built-in classification

5. Chroma
Speed: Fast for small-to-medium workloads
Cost: Free (open source)
Best for: Educational use, small RAGs
Limitations: Not built for production-scale retrieval yet

6. Milvus
Speed: Excellent (supports billions of vectors)
Cost: Free (community edition); paid cloud (Zilliz)
Best for: Large-scale systems (multi-modal), image/video/audio search
Organizations: Fintech, bioinformatics, autonomous systems

7. Redis Vector
Speed: Real-time optimized
Cost: Free open source; paid Redis Enterprise
Best for: If Redis is already part of your infra (e.g., caching, real-time apps)
Limitations: Lacks advanced vector indexing options

Which Vector Store Should You Use?
Use FAISS when:
You’re prototyping or building a small RAG system
You want speed without infrastructure
You don’t need persistence across sessions

Use Pinecone when:
You need scalable, production-grade retrieval
You want a managed solution (no infra worries)
You’re working with large document sets

Use Qdrant or Weaviate when:
You want metadata filtering
You want self-hosted + production-grade performance
You want to store documents + embeddings + metadata in one place

Use Milvus when:
You're working on large, multi-modal data
You need high throughput and GPU support

Real-World Scenarios:
| Use Case                                | Vector Store        |
| --------------------------------------- | ------------------- |
| Internal policy RAG for small org       | FAISS / Chroma      |
| Customer support FAQ bot (medium scale) | Qdrant / Weaviate   |
| Public-facing RAG search engine         | Pinecone / Weaviate |
| GenAI for PDFs + image documents        | Qdrant / Milvus     |
| Prototyping with LangChain locally      | FAISS / Chroma      |

Summary Table:
| Feature              | FAISS | Pinecone | Qdrant | Weaviate | Chroma  | Milvus |
| -------------------- | ----- | -------- | ------ | -------- | ------- | ------ |
| Open Source          | Yes   | No       | Yes    | Yes      | Yes     | Yes    |
| Cloud Hosted Option  | No    | Yes      | Yes    | Yes      | Limited | Yes    |
| Metadata Filtering   | No    | Yes      | Yes    | Yes      | Limited | Yes    |
| Multi-modal Support  | No    | No       | Yes    | Limited  | No      | Yes    |
| Persistence Built-in | No    | Yes      | Yes    | Yes      | Yes     | Yes    |
| Production Ready     | No    | Yes      | Yes    | Yes      | No      | Yes    |

"""

"\nWhat is a Vector Store?\nA vector store (or vector database) is a specialized storage engine that:\n-Stores high-dimensional embeddings (vectors)\n-Supports similarity search (e.g., cosine similarity, dot product)\n-Returns the most relevant stored chunks given a query\n\nUsed in:\n-Semantic search\n-RAG systems (retrieval-augmented generation)\n-Recommendation engines\n-Image/audio search\n\nCommon Vector Stores in the Industry (Open-source + Paid)\n| Name                     | Type                     | Open Source | Cloud Hosted           | Common Use Case                       | Notes                                   |\n| ------------------------ | ------------------------ | ----------- | ---------------------- | ------------------------------------- | --------------------------------------- |\n| **FAISS**                | In-memory                | Yes         | No                     | Fast local prototyping                | Lightweight, fast, no persistence       |\n| **Chro

In [22]:
# Step6: Initialize GPT-4 Turbo via LangChain
llm = ChatOpenAI(
    model="gpt-4-turbo" # Use the GPT-4 Turbo model for faster and more cost-effective response # Pass the OpenAI API key for authentication
)

# For demo purpose
# If we have to use Claude 3, the code will be as below
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(
#     model="claude-3",
#     api_key=openai_api_key
# )
# If you want to use Ollama, the code will be as below
# from langchain_ollama import Ollama
# llm = Ollama(model="llama2", base_url="http://localhost:11434") # Assuming Ollama is running locally
# If you want to use Gemini Pro, the code will be as below
# from langchain_gemini import Gemini
# llm = Gemini(model="gemini-pro", api_key=openai_api_key)

In [30]:
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI

# Initialize LLM
llm = ChatOpenAI(
    temperature=0,
    model="gpt-3.5-turbo"
)

# Create retriever from vectorstore
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

# Create RAG QA chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)


'\nWhat Happens Behind the Scenes in RetrievalQA:\n-User provides a question → rag_chain.invoke({"query": "..."})\n-retriever fetches top k matching chunks from FAISS\n-LangChain injects those chunks into the prompt\n-llm (e.g., GPT-4) uses that context to generate a grounded answer\n-Optionally, return_source_documents=True lets you see the supporting evidence\n'

In [31]:
def ask_pdf_agent(query):
    print(f"\nUser Query: {query}")

    result = rag_chain.invoke({
        "query": query
    })

    return result["result"]


# Sample Test
print(ask_pdf_agent("What does SPIL stand for?"))


In [32]:
# Step 9: Run a sample query
ask_pdf_agent("What does SPIL stand for?")


User Query: What does SPIL stand for?


TypeError: 'OpenAIEmbeddings' object is not callable

In [ ]:
ask_pdf_agent("How many leaves per month is an employee eligible for?")


User Query: How many leaves per month is an employee eligible for?


Answer:
An employee is eligible for 2 leaves per month.

Retrieved Passages:
--> g) Employees who are on “Official Duty” (OD) or “On Tour” (OT) are requested to take the 
written approval in advance from their Reporting Officer and submit the same to the HR 
Department for record  ...
--> i) The Leave given to the employee will be counted on Financial Year (i.e. April to March) and the 
balance as on March will be credited to the next financial year of employee leave balance 
account.  ...
--> completion of month. The leave given to the employee will be strictly based on the Date of 
Joining (i.e. Pro-Rata Basis). Employee who joins on the following dates will be eligible for the 
leave.  
 ...


In [ ]:
ask_pdf_agent("What does the acronym “RESPECT” in SPIL’s sales vision stand for?")



User Query: What does the acronym “RESPECT” in SPIL’s sales vision stand for?


Answer:
The acronym "RESPECT" in SPIL's sales vision stands for:

- **R**: Reliability - You can count on us
- **E**: Excellence - Is our Standard
- **S**: Service - Customer First and accomplish the needs
- **P**: People - Serve People with Fairness & Firmness
- **E**: ?
- **C**: ?
- **T**: ?

Unfortunately, the descriptions for E, C, and T are not provided in the information I have.

Retrieved Passages:
--> To be one of the most respectable brands in the category through brand building initiatives, 
providing world class products with consistent, quality, leading to profitability and growth of 
everyone  ...
--> innovative and cost saving solution within their total production process. "Team Sirca” works had 
to understand their customer’s products and production processes to become their most reliable & 
dep ...
--> SPIL Corporate HR Policies  
  
  
  
 Respect of the core values, policies and proc

In [ ]:
ask_pdf_agent("What are the eligibility criteria and process for the Employee Children Merit Reward?")



User Query: What are the eligibility criteria and process for the Employee Children Merit Reward?


Answer:
To be eligible for the Employee Children Merit Award at Sirca Paints India Limited, an employee's child must achieve the following academic standards:

1. **Class V:** A cash reward of Rs 2500 will be given if the child secures marks above 85%.
2. **Class VIII:** A cash reward of Rs 3500 is awarded if the child secures marks above 80%.
3. **Class X:** A cash reward of Rs 5000 is given if the child secures marks above 80%.
4. **Class XII:** Rs 10000 is awarded as a cash reward if the child secures marks above 80%.

The process for receiving this award involves the following steps:
1. Achieve the required marks as specified for each class level.
2. The highest aggregate marks secured by children of employees will be considered.
3. Details or evidence of the child's academic achievements should likely be submitted to the HR department for verification, although specific submission 

In [ ]:
ask_pdf_agent("What is the stipend given if I join this company as an intern??")


User Query: What is the stipend given if I join this company as an intern??


Answer:
The stipend given to interns at the company is Rs 7000.

Retrieved Passages:
--> Internships Programme:  The Company can hir e the students from the Reputed Colleges 
depend upon the number of students required for the particular project. The stipend will be 
given to the student  ...
--> the Internships with their project details. After review of the respective department, the proper 
approval from the Management will be taken to process the Internship programme. The monetary 
benefit ...
--> Campus Placement – On Roll Jobs: While completing the education, the organization can hire 
the students as Management Trainee/Graduate Engineer Trainee/Junior Executives. The 
proper pay structure w  ...


In [ ]:
ask_pdf_agent("If I join this company after my graduation, what will be my annual package??")


User Query: If I join this company after my graduation, what will be my annual package??


Answer:
If you join the company after your graduation as a Management Trainee, Graduate Engineer Trainee, or Junior Executive, your annual salary package will be in the range of Rs 250,000 to 350,000. The exact amount within this range will be based on your performance in the interview.

Retrieved Passages:
--> Campus Placement – On Roll Jobs: While completing the education, the organization can hire 
the students as Management Trainee/Graduate Engineer Trainee/Junior Executives. The 
proper pay structure w  ...
--> Internships Programme:  The Company can hir e the students from the Reputed Colleges 
depend upon the number of students required for the particular project. The stipend will be 
given to the student  ...
--> And based upon the performance of the candidate after a year, the Management 
Trainee/Graduate Engineer Trainee /Junior Executives will be promoted on the “Executive” 
Grade.  


In [ ]:
ask_pdf_agent("Is this Company strict with the office timings??")


User Query:  is this Company strict with the office timings??


Answer:
The company does offer flexible working hours, allowing employees to choose between starting at 9:30 AM and concluding at 6:00 PM, or starting at 10:00 AM and finishing at 6:30 PM. There is also a 15-minute grace period for lateness. However, if an employee arrives later than these grace periods, he or she will face deductions from their leave balance. The policy also mentions that in extreme emergencies, employees are allowed to arrive as late as 11:45 AM. This shows that while there is some flexibility, the company does maintain clear rules on timing to ensure productivity is not compromised.

Retrieved Passages:
--> Every employee in the company has an important role in ensuring the smooth and efficient flow 
of daily business act ivities. The purpose of attendance is to provide the clarity to all employees 
abou ...
--> SPIL Corporate HR Policies  
 
 
 
c) For the betterment and work life balance, the organiz


# Conversational RAG with Memory

This section adds conversational memory support using:

- `ConversationBufferMemory`
- `ConversationalRetrievalChain`

The old `RetrievalQA` pipeline is preserved.


In [ ]:

from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain


In [ ]:

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

print("Memory initialized successfully")


In [ ]:

conversation_rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    verbose=True
)

print("Conversational RAG chain created successfully")


In [ ]:

response1 = conversation_rag_chain.invoke({
    "question": "Who is the candidate?"
})

print("\nAnswer 1:\n")
print(response1["answer"])


response2 = conversation_rag_chain.invoke({
    "question": "What are the candidate's technical skills?"
})

print("\nAnswer 2:\n")
print(response2["answer"])


response3 = conversation_rag_chain.invoke({
    "question": "Does the candidate know deep learning?"
})

print("\nAnswer 3:\n")
print(response3["answer"])


In [ ]:

print("=== Chat History ===")
print(memory.buffer)



# Advanced Conversational RAG Demo

This notebook now contains:

1. Basic RAG (Without Memory)
2. Conversational RAG (With Memory)
3. Conversational RAG with Condensed Question Memory

The purpose is to compare:
- retrieval quality
- follow-up question handling
- conversational continuity


In [ ]:

from langchain.prompts import PromptTemplate


In [ ]:

CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(
'''
Given the following conversation and a follow up question,
rephrase the follow up question to be a standalone question.

Chat History:
{chat_history}

Follow Up Input:
{question}

Standalone question:
'''
)

print("Condense question prompt created successfully")


In [ ]:

advanced_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key='answer'
)

print("Advanced memory initialized")


In [ ]:

advanced_conversation_rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=advanced_memory,
    condense_question_prompt=CONDENSE_QUESTION_PROMPT,
    return_source_documents=True,
    verbose=True
)

print("Advanced conversational RAG chain created")



# Demo Questions

## Demo Without Memory

```python
query = "What is the probation period?"
response = rag_chain.invoke(query)
print(response)
```

Follow-up:

```python
query = "Can it be extended?"
response = rag_chain.invoke(query)
print(response)
```

Observation:
- Follow-up questions may fail because there is no chat history.

---

# Demo With Memory

```python
response1 = conversation_rag_chain.invoke({
    "question": "What is the probation period?"
})

print(response1["answer"])
```

```python
response2 = conversation_rag_chain.invoke({
    "question": "Can it be extended?"
})

print(response2["answer"])
```

Observation:
- Better conversational continuity.

---

# Demo With Condensed Question Memory

```python
response1 = advanced_conversation_rag_chain.invoke({
    "question": "What is the probation period?"
})

print(response1["answer"])
```

```python
response2 = advanced_conversation_rag_chain.invoke({
    "question": "Can it be extended?"
})

print(response2["answer"])
```

Observation:
- Follow-up question gets reformulated internally.
- Retrieval quality becomes significantly better.


In [ ]:

# Advanced Conversational Demo

questions = [
    "What is the probation period?",
    "Can it be extended?",
    "What happens if performance is poor?",
    "Who evaluates the employee?"
]

for q in questions:
    print("\n" + "="*80)
    print("QUESTION:", q)

    response = advanced_conversation_rag_chain.invoke({
        "question": q
    })

    print("\nANSWER:")
    print(response["answer"])



# Architecture Comparison

## 1. Basic RAG

User Question
↓
Retriever
↓
LLM
↓
Answer

Limitation:
- No conversational understanding

---

## 2. Conversational RAG with Memory

User Question
↓
Chat Memory
↓
Retriever
↓
LLM
↓
Answer

Advantage:
- Maintains conversation flow

---

## 3. Conversational RAG with Condensed Questions

User Question
↓
Chat History
↓
Question Rewriter
↓
Retriever
↓
LLM
↓
Answer

Advantage:
- Converts ambiguous follow-up questions
  into standalone questions before retrieval.
